# Tarea 2 · Parte 2. Modelos sobre la muestra de evaluación (Google Colab)

Este notebook corre todos los modelos que se comparan en `02_evaluacion.ipynb` sobre **los mismos 10 documentos y 10 clips**, definidos en `referencias.py`. La laptop, con 7,7 GB de RAM y sin GPU, no alcanzaba, así que se corrió en Colab con una GPU T4. Las configuraciones se fijaron antes de calcular cualquier métrica y no se ajustaron después.

| Modalidad | Modelo | Rol | Dispositivo |
|---|---|---|---|
| Documentos | PaddleOCR 3.7.0 por defecto (`lang="es"`, sin cambiar nada más) | línea base | CPU |
| Documentos | Tesseract 5 (`spa`, `--psm 3`) | OCR alternativo | CPU |
| Audio | faster-whisper `small` int8, `language="es"`, beam 5, VAD | línea base | CPU |
| Audio | faster-whisper `large-v3-turbo` float16, mismos parámetros | modelo mejorado | GPU |
| Audio | Meta MMS `mms-1b-all`, adaptador `spa` (CTC, ventanas de 30 s) | ASR alternativo | GPU |

La configuración de PaddleOCR de la Parte 1, con detector liviano y sin modelos de orientación, ya tiene sus salidas en el repositorio y entra a la comparación como tercera opción de OCR.

PaddleOCR va en CPU a propósito, porque la versión con GPU choca con el PyTorch que necesitan los modelos de audio. Para el OCR no se mide la velocidad, solo el CER.

Al final descarga `modelos_resultados.zip`, que se copia a `tarea_2/datos/evaluacion/`.

Las salidas de la ejecución en Colab no quedaron en esta copia. Sus resultados están en `datos/evaluacion/` y se analizan en `02_evaluacion.ipynb`.

In [ ]:
# 1. Entorno
!apt-get -qq install -y tesseract-ocr tesseract-ocr-spa > /dev/null
!pip -q install paddlepaddle==3.2.2 paddleocr==3.7.0 pytesseract faster-whisper==1.2.1 jiwer
# faster-whisper (CTranslate2) usa cuBLAS y cuDNN de CUDA 12; Colab ya trae CUDA 13
!pip -q install nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"
import subprocess
print(subprocess.run(["tesseract", "--version"], capture_output=True, text=True).stdout.splitlines()[0])
print(subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout)

In [ ]:
# 2. Muestra, bajada desde Wikimedia Commons con descargar.py y verificada contra la local
import hashlib, sys, urllib.request
from pathlib import Path
from PIL import Image
BASE = "https://raw.githubusercontent.com/nicosil02/Python-Intermedio/parte-2-evaluacion/tarea_2/"
for f in ("descargar.py", "referencias.py"):
    urllib.request.urlretrieve(BASE + f, f)
sys.path.insert(0, ".")
import descargar as d
from referencias import MUESTRA_DOCS, MUESTRA_CLIPS

ESPERADO = {
 "doc_002": "13fb4aba56bc5c211b75c102ca4351622ea13328a21c50439c90858fd62275f5",
 "doc_003": "67bd210b3d32db42841031fbe9961990e0d1fdb7348e40f1cc83ad7526ce0a85",
 "doc_004": "b89fb831a26942eb444eeb83dfe3841a6c828b8c642f12221ff5315a49e8845c",
 "doc_009": "b14555f8d9d3dc094f153489b6ac11866fc9b41c793b45c6fb4810e379a14788",
 "doc_013": "361d6b4b4a0370af5d9fb59c7a7c722aec7c827a07d47f8920f3de23e7a213ad",
 "doc_016": "df7110e1894701c8ec5093c2b1cc92a0ba309a6d8cde928c78e489a5eecf8c90",
 "doc_018": "250ccfd1dc8534b15f0cf01a0e9603f7252155340754c54c08272cf68b5e4168",
 "doc_021": "6d2eaea032c9ada1802941fb6d55c558f8181c2e3cbd15264c812b5f3bda6576",
 "doc_025": "89e6214ad92ed26ebb59f944e2bf24e0528395d9b0e43c305beca826bcd6e70a",
 "doc_022": "fa63b411cc39cbef873c345c6e5583bed142566a619e53522ce7a1b2e84b4e42",
 "clip_001": "4650e7cebe4d4dda1da8b1ff618f05d35b3f2532cc3121596250dce094445bcc",
 "clip_007": "018fd1b93ca94e6806d519c2558cee46d74234378ce3a1e5ad1f5970dce8985f",
 "clip_010": "294cefd36add195773f3b94b99e44262944a513043564345c8f350f6e07ddce9",
 "clip_012": "754c9feb96d3bba25712ae9c734c845715995ee7eea5adc4e75805d1e6263f65",
 "clip_013": "b1506066acb3e4c5bb8bae67fc81403b2868fb5479e4796f3ecb8d3322859557",
 "clip_014": "e89645a9436c994d8d28fd6c77338ee2a955ab854df48eef864aa1576d048147",
 "clip_015": "2c22a04eceeea612c094fab677fa91614324275d457e484bb4a336e40843756e",
 "clip_016": "80f0595599c10d49306e0e2a5841e54355372928c48f86b146ea9f285901040a",
 "clip_018": "7446447e49c92d467112b120284681f11fcef45a90948e42983222721b2ffde1",
 "clip_023": "c45d1999fa0cf98c9f4eae69b1d0dcf7ea6976c03960219d1c37afc20ab25fe4"
}
docs_dir, aud_dir = Path("documentos"), Path("audio")
docs_dir.mkdir(exist_ok=True); aud_dir.mkdir(exist_ok=True)
todos = {**d.DOCUMENTOS, **d.RESERVAS}
for id_ in MUESTRA_DOCS:
    tipo, titulo, pagina, recorte = todos[id_]
    destino = docs_dir / f"{id_}.jpg"
    d.bajar(d.ficha(titulo, pagina)["url"], destino)
    if recorte:
        im = Image.open(destino)
        im.crop(tuple(int(v * t) for v, t in zip(recorte, im.size * 2))).save(destino, quality=95)
for id_ in MUESTRA_CLIPS:
    titulo = d.AUDIOS[id_]
    destino = aud_dir / f"{id_}{Path(titulo).suffix.lower()}"
    d.bajar(d.ficha(titulo)["url"], destino)
archivos = {p.stem: p for p in [*docs_dir.glob("*.jpg"), *aud_dir.glob("*")]}
distintos = [i for i, h in ESPERADO.items() if hashlib.sha256(archivos[i].read_bytes()).hexdigest() != h]
print(f"{len(ESPERADO) - len(distintos)}/{len(ESPERADO)} archivos idénticos a los locales", distintos or "")

In [ ]:
%%writefile ocr_eval.py
# 3. OCR con PaddleOCR por defecto (línea base) y Tesseract
# Va en un proceso aparte porque PaddlePaddle y PyTorch en el mismo proceso hacen caer la sesión de Colab.
import json, time
from pathlib import Path
import numpy as np, paddle, paddleocr, pytesseract
from PIL import Image
from paddleocr import PaddleOCR
from referencias import MUESTRA_DOCS
docs_dir, salida = Path("documentos"), Path("resultados")
salida.mkdir(exist_ok=True)
paddle.set_device("cpu")
print("PaddleOCR", paddleocr.__version__, "| PaddlePaddle", paddle.__version__)
ocr = PaddleOCR(lang="es")
res = {}
for id_ in MUESTRA_DOCS:
    t0 = time.perf_counter()
    r = ocr.predict(str(docs_dir / f"{id_}.jpg"))[0]
    res[id_] = {"texto": chr(10).join(r["rec_texts"]), "confianza": float(np.mean(r["rec_scores"])) if r["rec_scores"] else 0.0,
                "segundos": time.perf_counter() - t0}
    print(id_, len(res[id_]["texto"]), "caracteres", round(res[id_]["segundos"]), "s")
(salida / "ocr_paddle_base.json").write_text(json.dumps(res, ensure_ascii=False, indent=1), encoding="utf-8")

res = {}
for id_ in MUESTRA_DOCS:
    t0 = time.perf_counter()
    texto = pytesseract.image_to_string(Image.open(docs_dir / f"{id_}.jpg"), lang="spa", config="--psm 3")
    res[id_] = {"texto": texto.strip(), "segundos": time.perf_counter() - t0}
    print(id_, len(res[id_]["texto"]), "caracteres", round(res[id_]["segundos"], 1), "s")
(salida / "ocr_tesseract.json").write_text(json.dumps(res, ensure_ascii=False, indent=1), encoding="utf-8")
(salida / "entorno_ocr.json").write_text(json.dumps({"paddleocr": paddleocr.__version__, "paddlepaddle": paddle.__version__}), encoding="utf-8")

In [ ]:
!python ocr_eval.py 2>&1 | grep -v -i "warn\|download\|fetching\|reconstruction\|creating model\|using official"


In [ ]:
# 4. ASR con faster-whisper small (CPU, línea base) y large-v3-turbo (GPU)
import gc, json, time, torch, librosa
from faster_whisper import WhisperModel
salida = Path("resultados"); salida.mkdir(exist_ok=True)
audios = {p.stem: librosa.load(p, sr=16000, mono=True)[0] for p in sorted(aud_dir.iterdir())}
PARAMS = dict(language="es", beam_size=5, vad_filter=True)

# Carga las librerías de CUDA 12 instaladas con pip, para que CTranslate2 las encuentre
import ctypes, glob, site
libs = glob.glob(site.getsitepackages()[0] + "/nvidia/cublas/lib/*.so.12") + glob.glob(site.getsitepackages()[0] + "/nvidia/cudnn/lib/*.so.9")
for _ in range(3):  # algunas dependen de otras: se reintenta hasta cargar todas
    for lib in list(libs):
        try:
            ctypes.CDLL(lib, mode=ctypes.RTLD_GLOBAL); libs.remove(lib)
        except OSError:
            pass

def correr_whisper(nombre, modelo, device, compute_type):
    m = WhisperModel(modelo, device=device, compute_type=compute_type)
    res = {}
    for id_, a in audios.items():
        t0 = time.perf_counter()
        segs, _ = m.transcribe(a, **PARAMS)
        texto = " ".join(s.text.strip() for s in segs)
        seg = time.perf_counter() - t0
        res[id_] = {"texto": texto, "segundos": seg, "duracion_seg": len(a) / 16000, "rtf": seg / (len(a) / 16000)}
        print(nombre, id_, f"RTF {res[id_]['rtf']:.2f}")
    (salida / f"asr_{nombre}.json").write_text(json.dumps(res, ensure_ascii=False, indent=1), encoding="utf-8")
    del m; gc.collect(); torch.cuda.empty_cache()

correr_whisper("whisper_small_cpu", "small", "cpu", "int8")
correr_whisper("whisper_turbo_gpu", "large-v3-turbo", "cuda", "float16")

In [ ]:
# 5. ASR alternativo con Meta MMS (mms-1b-all, español)
from transformers import pipeline
mms = pipeline("automatic-speech-recognition", model="facebook/mms-1b-all", device=0, torch_dtype=torch.float16,
               model_kwargs={"target_lang": "spa", "ignore_mismatched_sizes": True})
res = {}
for id_, a in audios.items():
    t0 = time.perf_counter()
    texto = mms(a.copy(), chunk_length_s=30, stride_length_s=5)["text"]
    seg = time.perf_counter() - t0
    res[id_] = {"texto": texto, "segundos": seg, "duracion_seg": len(a) / 16000, "rtf": seg / (len(a) / 16000)}
    print("mms", id_, f"RTF {res[id_]['rtf']:.3f}", texto[:80])
(salida / "asr_mms_gpu.json").write_text(json.dumps(res, ensure_ascii=False, indent=1), encoding="utf-8")

In [ ]:
# 6. Versiones usadas y descarga de resultados
import faster_whisper, transformers, platform
(salida / "entorno.json").write_text(json.dumps({
    **json.loads((salida / "entorno_ocr.json").read_text()),
    "tesseract": subprocess.run(["tesseract", "--version"], capture_output=True, text=True).stdout.splitlines()[0],
    "faster_whisper": faster_whisper.__version__, "transformers": transformers.__version__, "torch": torch.__version__,
    "gpu": torch.cuda.get_device_name(0), "cpu_nucleos": __import__("os").cpu_count(), "python": platform.python_version()},
    indent=1), encoding="utf-8")
!cd resultados && zip -q -r ../modelos_resultados.zip .
from google.colab import files
files.download("modelos_resultados.zip")